# OpsGuardian: Evidence-Grounded DevOps Incident Resolution Assistant

**Kushagra Gupta - RAG Portfolio Project**

This notebook intentionally follows the syntax and workflow of Sir's six in-class Generative AI labs. The custom addition is only the DevOps dataset, metadata and incident-specific output schema.

## What from Sir's labs is reused?

| Lab | OpsGuardian use |
|---|---|
| 1 - LLM Generation Parameters | Low-temperature factual generation |
| 2 - Introduction to LangChain | Chat model + ChatPromptTemplate + runnable chain |
| 3 - Structured Output | Pydantic + `with_structured_output` |
| 4 - RAG | loaders, 1000/200 split, OllamaEmbeddings, Chroma, BM25, EnsembleRetriever |
| 5 - Advanced RAG | MMR + MultiQueryRetriever |
| 6 - RAG With LangGraph | `START -> retrieve -> generate -> END` |

## 1. Initial Setup

In [1]:
import os, warnings
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from uuid import uuid4
from IPython.display import display, Markdown
warnings.filterwarnings('ignore')
load_dotenv(override=True)
NOTEBOOK_DIR=Path.cwd(); PROJECT_ROOT=NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name=='notebooks' else NOTEBOOK_DIR
DATASET_DIR=PROJECT_ROOT/'dataset'; EVAL_FILE=PROJECT_ROOT/'evaluation'/'evaluation_questions.csv' 

### 1.1 Chat model - Lab 1 / Lab 2 pattern

In [2]:
from langchain.chat_models import init_chat_model
from langchain_ollama import ChatOllama
provider=os.getenv('MODEL_PROVIDER','ollama').lower()
if provider == 'groq':
    llm=init_chat_model(os.getenv('GROQ_CHAT_MODEL','openai/gpt-oss-20b'), model_provider='groq', temperature=0.1)
else:
    llm=ChatOllama(model=os.getenv('OLLAMA_CHAT_MODEL','qwen3:8b'), temperature=0.1)
llm

ChatOllama(model='qwen3:8b', temperature=0.1)

### 1.2 Embedding model - Lab 4 / Lab 5 pattern

In [3]:
from langchain_ollama import OllamaEmbeddings
embeddings_model=OllamaEmbeddings(model=os.getenv('OLLAMA_EMBED_MODEL','nomic-embed-text'))
embeddings_model

OllamaEmbeddings(model='nomic-embed-text', validate_model_on_init=False, base_url=None, client_kwargs={}, async_client_kwargs={}, sync_client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, keep_alive=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

## 2. Custom Dataset

In [4]:
manifest_df=pd.read_csv(DATASET_DIR/'source_manifest.csv').fillna('')
print('Sources:',len(manifest_df)); display(manifest_df.head(10)); print(manifest_df['category'].value_counts())

Sources: 34


,source_path,category,service,incident_id,title
0,runbooks/api_latency.md,runbooks,checkout-api,,Api Latency
1,runbooks/redis_timeout.md,runbooks,platform,,Redis Timeout
2,runbooks/db_pool.md,runbooks,platform,,Db Pool
3,runbooks/crashloop.md,runbooks,platform,,Crashloop
4,runbooks/oom.md,runbooks,platform,,Oom
5,runbooks/dns.md,runbooks,platform,,Dns
6,runbooks/tls_certificate.md,runbooks,platform,,Tls Certificate
7,runbooks/queue_backlog.md,runbooks,platform,,Queue Backlog
8,runbooks/auth_401.md,runbooks,platform,,Auth 401
9,docs/system_architecture.md,docs,checkout-api,,System Architecture


category
runbooks       9
incidents      6
configs        4
logs           4
docs           3
postmortems    3
alerts         3
k8s            2
Name: count, dtype: int64


## 3. Load Documents

In [5]:
from ops_loader import load_ops_dataset
all_docs=load_ops_dataset(DATASET_DIR)
print('Loaded documents:',len(all_docs)); print(all_docs[0].metadata); print(all_docs[0].page_content[:500])

Loaded documents: 34
{'source': 'runbooks/api_latency.md', 'title': 'Api Latency', 'category': 'runbooks', 'service': 'checkout-api', 'incident_id': ''}
# Runbook: API Latency Spike
Service scope: checkout-api, payments-api, orders-api.

Trigger: p95 latency above 800 ms for 10 minutes.

Triage order:
1. Compare current latency with the last deployment time.
2. Check upstream dependency latency before restarting pods.
3. Inspect Redis, database, and payment-gateway timeout counters.
4. Check Kubernetes CPU throttling and memory pressure.
5. If a new release correlates with the spike and dependency health is normal, perform a controlled rollback.


## 4. Split Documents - same 1000 / 200 baseline as Lab 4

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200,add_start_index=True)
all_splits=text_splitter.split_documents(all_docs)
print(f'Split {len(all_docs)} documents into {len(all_splits)} chunks.')

Split 34 documents into 34 chunks.


## 5. Store Chunks in Chroma - Lab 4 / Lab 5

In [7]:
from langchain_chroma import Chroma
PERSIST_DIR=str(PROJECT_ROOT/'chroma_opsguardian_db')
vector_store_chroma=Chroma(collection_name='opsguardian-rag',embedding_function=embeddings_model,persist_directory=PERSIST_DIR)
uuids=[str(uuid4()) for _ in range(len(all_splits))]
# Run once on a clean index. Delete chroma_opsguardian_db before rebuilding to avoid duplicates.
vector_store_chroma.add_documents(documents=all_splits,ids=uuids)
print('Indexed chunks:',len(all_splits))

Indexed chunks: 34


## 6. Retrieval Methods from Class

In [8]:
def get_similarity_retriever(service='all',k=4):
    kwargs={'k':k}
    if service not in ('all','platform'): kwargs['filter']={'service':service}
    return vector_store_chroma.as_retriever(search_type='similarity',search_kwargs=kwargs)

def get_mmr_retriever(service='all',k=4):
    kwargs={'k':k}
    if service not in ('all','platform'): kwargs['filter']={'service':service}
    return vector_store_chroma.as_retriever(search_type='mmr',search_kwargs=kwargs)

In [9]:
from langchain.retrievers import BM25Retriever, EnsembleRetriever

def get_service_splits(service='all'):
    if service in ('all','platform'): return all_splits
    selected=[d for d in all_splits if d.metadata.get('service') in (service,'platform')]
    return selected or all_splits

def get_hybrid_retriever(service='all',k=4):
    selected_docs=get_service_splits(service)
    bm25_retriever=BM25Retriever.from_documents(documents=selected_docs); bm25_retriever.k=k
    chroma_retriever=get_similarity_retriever(service,k)
    return EnsembleRetriever(retrievers=[bm25_retriever,chroma_retriever],weights=[0.7,0.3])

In [10]:
service='checkout-api'; question='Why did checkout latency spike after v2.18.0?'
retrieved_docs=get_hybrid_retriever(service).invoke(question)
for i,d in enumerate(retrieved_docs,1): print(i,d.metadata.get('source'),d.page_content[:300],'\n')

1 incidents/INC-001-checkout-redis.md # INC-001 - Checkout Latency after v2.18.0
Severity: SEV-2
Service: checkout-api
Start: 2026-08-14 10:02 UTC

Impact: checkout p95 rose from 340 ms to 2.8 s; 7% of requests returned 504.
Evidence: `RedisTimeoutError` began within three minutes of deployment v2.18.0. Redis server CPU was 31% and heal 

2 runbooks/api_latency.md # Runbook: API Latency Spike
Service scope: checkout-api, payments-api, orders-api.

Trigger: p95 latency above 800 ms for 10 minutes.

Triage order:
1. Compare current latency with the last deployment time.
2. Check upstream dependency latency before restarting pods.
3. Inspect Redis, database, and 

3 runbooks/auth_401.md # Runbook: Sudden 401 Increase
Check token validation errors, issuer/audience configuration, key rotation, and clock synchronization. A broad 401 spike immediately after a config deployment often indicates auth configuration mismatch. Do not reset user passwords without evidence. 

4 runbooks/redis_timeout

## 7. Prompt - Lab 4 ChatPromptTemplate

In [11]:
from langchain_core.prompts import ChatPromptTemplate
system_message_template="""You are OpsGuardian, a DevOps incident-resolution assistant.
Use ONLY the retrieved context. Distinguish evidence from possible causes. Do not invent metrics, incidents or remediation.
Prefer reversible diagnostics. If evidence is insufficient, say so clearly.
Selected service: {service}
Retrieved context:
{context}
"""
ops_prompt=ChatPromptTemplate([('system',system_message_template),('human','Incident question: {question}')])

## 8. Structured Output - Lab 3

In [12]:
from pydantic import BaseModel,Field
class OpsGuardianResponse(BaseModel):
    incident_summary:str=Field(description='Short evidence-grounded incident summary')
    likely_causes:list[str]=Field(description='Likely causes supported by context')
    recommended_steps:list[str]=Field(description='Safe ordered next steps')
    evidence:list[str]=Field(description='Concrete evidence from context')
    source_documents:list[str]=Field(description='Sources used')
    escalation_required:bool=Field(description='Whether evidence indicates escalation')
    information_gap:bool=Field(description='Whether retrieved evidence is insufficient')
structured_llm=llm.with_structured_output(OpsGuardianResponse)

## 9. LCEL RAG Chain - same idea as Lab 4

In [13]:
def format_docs(docs):
    return '\n\n'.join([f"SOURCE: {d.metadata.get('source')}\nSERVICE: {d.metadata.get('service')}\nCATEGORY: {d.metadata.get('category')}\nCONTENT: {d.page_content}" for d in docs])

def ask_opsguardian_lcel(service,question):
    docs=get_hybrid_retriever(service).invoke(question)
    answer=(ops_prompt|structured_llm).invoke({'service':service,'question':question,'context':format_docs(docs)})
    return answer,docs

In [14]:
# Lab 2/3 demo: LCEL chain with structured output.
answer, evidence = ask_opsguardian_lcel('checkout-api', 'Why did checkout latency spike after v2.18.0?')
print(answer.model_dump())


{'incident_summary': 'Checkout latency spiked after deploying v2.18.0 due to a **client connection-pool limit reduction** from 80 to 20. This caused RedisTimeoutError and connection exhaustion, leading to 504 errors and elevated p95 latency (2.8s vs. 340ms). Rollback to v2.17.4 resolved the issue within six minutes.', 'likely_causes': ['Accidental reduction of Redis client connection-pool size in v2.18.0 (from 80 to 20).', 'RedisPoolActive metric reached the new limit (20), while waiters rose to 46, indicating connection exhaustion.'], 'recommended_steps': ['Verify Redis client configuration for connection-pool settings in new deployments.', 'Monitor RedisPoolActive and RedisTimeoutError metrics during rollouts.', 'Follow the Redis Timeout runbook to correlate deployment timing with timeout spikes.'], 'evidence': ['RedisTimeoutError occurred within three minutes of v2.18.0 deployment.', 'redis_pool_active=20 (new limit) vs. previous 80.', 'Rollback to v2.17.4 restored normal latency.']

## 10. Advanced RAG Experiment - MultiQuery from Lab 5

In [15]:
from langchain.retrievers.multi_query import MultiQueryRetriever
base_retriever=get_similarity_retriever('checkout-api',k=2)
mq_retriever=MultiQueryRetriever.from_llm(retriever=base_retriever,llm=llm,include_original=True)
# mq_docs=mq_retriever.invoke('checkout got very slow right after deploy')

## 11. Final Workflow with LangGraph - Lab 6

In [16]:
from typing_extensions import TypedDict
from typing import List
from langchain_core.documents import Document
from langgraph.graph import START,StateGraph,END
class State(TypedDict):
    service:str
    question:str
    context:List[Document]
    answer:OpsGuardianResponse

def retrieve(state:State):
    return {'context':get_hybrid_retriever(state['service']).invoke(state['question'])}

def generate(state:State):
    response=(ops_prompt|structured_llm).invoke({'service':state['service'],'question':state['question'],'context':format_docs(state['context'])})
    return {'answer':response}

graph_builder=StateGraph(State); graph_builder.add_node('retrieve',retrieve); graph_builder.add_node('generate',generate)
graph_builder.add_edge(START,'retrieve'); graph_builder.add_edge('retrieve','generate'); graph_builder.add_edge('generate',END)
graph=graph_builder.compile()

C:\Users\Eldoria\Pictures\portfolio projects\OpsGuardian_End_to_End_Sir_Code\OpsGuardian_End_to_End_Sir_Code\.venv\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [17]:
# Lab 6 demo: the same style of question through the LangGraph pipeline.
result = graph.invoke({'service': 'payments-api', 'question': 'Why were payment requests failing with PoolTimeout?'})
print(result['answer'].model_dump())


{'incident_summary': 'Payment requests failed with PoolTimeout due to database connection pool saturation.', 'likely_causes': ['Connection leak during exception handling introduced in release v4.6.1', 'Gateway timeout path failed to release connections, exhausting the pool'], 'recommended_steps': ['Verify connection pool metrics: active=30, idle=0, waiters=118', 'Check for recent code changes around exception handling and connection cleanup', 'Monitor database CPU (42%) and max_connections headroom (120 available)'], 'evidence': ['PoolTimeout logs with waiters=118', 'Release v4.6.1 introduced code path failing to release connections on gateway timeout', 'Database CPU at 42% with max_connections headroom'], 'source_documents': ['incidents/INC-002-payments-db-pool.md', 'logs/payments.log'], 'escalation_required': False, 'information_gap': False}


## 12. Retrieval Evaluation - Similarity vs MMR vs Hybrid

In [18]:
eval_df=pd.read_csv(EVAL_FILE).fillna(''); display(eval_df.head())
def source_hit(docs,expected): return int(any(d.metadata.get('source')==expected for d in docs))
def category_hit(docs,expected): return int(any(d.metadata.get('category')==expected for d in docs))
def evaluate_retriever(method='hybrid'):
    rows=[]
    for _,row in eval_df[eval_df.expected_source!=''].iterrows():
        r=get_similarity_retriever(row.service) if method=='similarity' else get_mmr_retriever(row.service) if method=='mmr' else get_hybrid_retriever(row.service)
        docs=r.invoke(row.question)
        rows.append({'question_id':row.question_id,'method':method,'source_hit':source_hit(docs,row.expected_source),'category_hit':category_hit(docs,row.expected_category),'sources':' | '.join(d.metadata.get('source','') for d in docs)})
    return pd.DataFrame(rows)
# similarity_results=evaluate_retriever('similarity')
# mmr_results=evaluate_retriever('mmr')
# hybrid_results=evaluate_retriever('hybrid')

,question_id,service,question,expected_source,expected_category,expected_keyword
0,Q01,checkout-api,Why did checkout latency spike after v2.18.0?,incidents/INC-001-checkout-redis.md,incidents,redis
1,Q02,checkout-api,Should we flush Redis to solve the checkout in...,postmortems/PM-001-redis.md,postmortems,flush
2,Q03,checkout-api,What changed in the Redis pool between current...,configs/checkout.env.txt,configs,pool
3,Q04,payments-api,What caused PoolTimeout during the payments ou...,incidents/INC-002-payments-db-pool.md,incidents,connection
4,Q05,payments-api,Was database CPU the main cause of the payment...,logs/payments.log,logs,cpu


## 13. Controlled Failure Test

In [19]:
# Guardrail check: information_gap should be true for questions outside the corpus.
unsupported = graph.invoke({'service': 'all', 'question': "Who won yesterday's cricket match?"})
print(unsupported['answer'].model_dump())


{'incident_summary': 'The provided context contains no information about cricket matches or sports events. All logs, alerts, and postmortems pertain to DevOps infrastructure, Kubernetes services, and application performance issues. There is no evidence or mention of a cricket match outcome in the retrieved data.', 'likely_causes': ['The question is unrelated to the technical context provided.'], 'recommended_steps': ['Consult sports news sources or official cricket match reports for information on the match outcome.'], 'evidence': [], 'source_documents': [], 'escalation_required': False, 'information_gap': True}


## 14. Limitations

1. This bundled corpus is a portfolio/demo dataset, not a live production telemetry feed.
2. Real incident diagnosis requires current metrics, traces and permissions.
3. Retrieved historical incidents are evidence, not proof that a new incident has the same root cause.
4. Destructive remediation must remain human-approved.

## 15. AI Usage Disclosure
AI tools were used as a helper for project framing, code organization and debugging. The notebook implementation intentionally follows the instructor's Labs 1-6 patterns and should be understood and executed by the student.